# Notebook 07: Calculate Cap Index

Compute Cap Index and related log-transformed metrics.

**Critical Test #1:** Verify the Cap Index formula and investigate the
`log_Hub`/`log_FANTOM` columns.

**Confirmed formula:** `Cap_index = FANTOM_Total_CAGE_TPM / Cytosol_TPM`
(verified algebraically — e.g., GAPDH: 3238.908/1987.596 = 1.6296)

**Unresolved:** `log_Hub` and `log_FANTOM` columns do NOT correspond to simple
log10 transforms of visible columns. They appear to derive from a separate
Hub CAGE dataset or from CPM columns. These are NOT used in the scatter plots
(which use `Cap_index`), so they are lower priority.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../reference")

## 1. Load Merged Data

In [2]:
merged = pd.read_csv(PROCESSED_DIR / "06_hub_cage_merged.csv")
print(f"Loaded {len(merged):,} merged genes")

Loaded 12,811 merged genes


## 2. Calculate Cap Index

`Cap_index = FANTOM_Total_CAGE_TPM / Cytosol_TPM`

When Cytosol_TPM = 0, Cap_index = 0 (matching Jason's behavior).

In [3]:
# Cap_index = FANTOM_Total_CAGE_TPM / Cytosol_TPM
# Handle division by zero: set to 0 when Cytosol_TPM = 0
merged["Cap_index"] = np.where(
    merged["Cytosol_TPM"] > 0,
    merged["FANTOM_Total_CAGE_TPM"] / merged["Cytosol_TPM"],
    0.0
)

print(f"Cap_index distribution:")
print(merged["Cap_index"].describe())
print(f"\nZero Cap_index: {(merged['Cap_index'] == 0).sum():,}")
print(f"Cap_index > 1: {(merged['Cap_index'] > 1).sum():,}")
print(f"Cap_index <= 0.1 (decapping candidates): {(merged['Cap_index'] <= 0.1).sum():,}")

Cap_index distribution:
count    12811.000000
mean         1.730863
std         20.247963
min          0.000000
25%          0.112136
50%          0.380825
75%          0.968020
max       1555.958838
Name: Cap_index, dtype: float64

Zero Cap_index: 1,923
Cap_index > 1: 3,121
Cap_index <= 0.1 (decapping candidates): 3,018


## 3. Investigate log_Hub and log_FANTOM Columns

Try candidate formulas against Jason's reference values to determine what
these columns represent.

In [4]:
ref = pd.read_csv(REFERENCE_DIR / "12544_Hub_CAGE_MERGE_with_CapIndex.csv")

# Inspect the reference log columns
print("Reference log columns (first 10 non-zero values):")
nonzero_log = ref[ref["log_Hub"] > 0].head(10)
for _, row in nonzero_log.iterrows():
    gene = row["Associated Gene Name"]
    print(f"  {gene}: log_Hub={row['log_Hub']:.6f}, log_FANTOM={row['log_FANTOM']:.6f}, "
          f"delta={row['delta_log10_Hub_minus_FANTOM']:.6f}")
    print(f"          CAGE_TPM={row['FANTOM_Total_CAGE_TPM']:.4f}, "
          f"Cytosol_TPM={row['Cytosol_TPM']:.4f}, "
          f"Cap_index={row['Cap_index']:.6f}")

Reference log columns (first 10 non-zero values):
  KCNQ3: log_Hub=0.039510, log_FANTOM=0.000000, delta=0.039510
          CAGE_TPM=0.0000, Cytosol_TPM=0.0068, Cap_index=0.000000
  MUC12: log_Hub=0.007453, log_FANTOM=0.000000, delta=0.007453
          CAGE_TPM=0.0000, Cytosol_TPM=0.0079, Cap_index=0.000000
  ZNF667: log_Hub=0.040381, log_FANTOM=0.117624, delta=-0.077243
          CAGE_TPM=0.3111, Cytosol_TPM=0.0079, Cap_index=39.354942
  MUC17: log_Hub=0.017323, log_FANTOM=0.000000, delta=0.017323
          CAGE_TPM=0.0000, Cytosol_TPM=0.0093, Cap_index=0.000000
  GPR158: log_Hub=0.047328, log_FANTOM=0.042846, delta=0.004481
          CAGE_TPM=0.1037, Cytosol_TPM=0.0093, Cap_index=11.102212
  TFEC: log_Hub=0.021790, log_FANTOM=0.000000, delta=0.021790
          CAGE_TPM=0.0000, Cytosol_TPM=0.0098, Cap_index=0.000000
  GRM8: log_Hub=0.164501, log_FANTOM=0.000000, delta=0.164501
          CAGE_TPM=0.0000, Cytosol_TPM=0.0098, Cap_index=0.000000
  CACNA1S: log_Hub=0.067558, log_FANTOM=0.00

In [5]:
# Test candidate formulas for log_Hub and log_FANTOM
# Work on genes where both log_Hub > 0 and log_FANTOM > 0
test = ref[(ref["log_Hub"] > 0) & (ref["log_FANTOM"] > 0)].copy()

candidates = {
    "log10(CAGE_TPM)": np.log10(test["FANTOM_Total_CAGE_TPM"].clip(lower=1e-10)),
    "log10(CAGE_TPM + 1)": np.log10(test["FANTOM_Total_CAGE_TPM"] + 1),
    "log10(Cytosol_TPM)": np.log10(test["Cytosol_TPM"].clip(lower=1e-10)),
    "log10(Cytosol_TPM + 1)": np.log10(test["Cytosol_TPM"] + 1),
    "log10(PB_TPM)": np.log10(test["PB_TPM"].clip(lower=1e-10)),
    "log10(PB_TPM + 1)": np.log10(test["PB_TPM"] + 1),
}

# Also try CPM columns if they exist
cpm_cols = [c for c in test.columns if 'count per million' in c.lower()]
if cpm_cols:
    avg_pb_cpm = [c for c in cpm_cols if 'sorted P-body' in c and 'average' in c]
    avg_cyt_cpm = [c for c in cpm_cols if 'pre-sorted' in c and 'average' in c]
    if avg_pb_cpm:
        candidates["log10(PB_avg_CPM)"] = np.log10(test[avg_pb_cpm[0]].clip(lower=1e-10))
        candidates["log10(PB_avg_CPM + 1)"] = np.log10(test[avg_pb_cpm[0]] + 1)
    if avg_cyt_cpm:
        candidates["log10(Cyt_avg_CPM)"] = np.log10(test[avg_cyt_cpm[0]].clip(lower=1e-10))
        candidates["log10(Cyt_avg_CPM + 1)"] = np.log10(test[avg_cyt_cpm[0]] + 1)

print("Testing candidate formulas against log_Hub:")
for name, vals in candidates.items():
    r, _ = stats.pearsonr(test["log_Hub"], vals)
    max_diff = (test["log_Hub"] - vals).abs().max()
    print(f"  {name:<30} r={r:.8f}  max_diff={max_diff:.6f}")

print("\nTesting candidate formulas against log_FANTOM:")
for name, vals in candidates.items():
    r, _ = stats.pearsonr(test["log_FANTOM"], vals)
    max_diff = (test["log_FANTOM"] - vals).abs().max()
    print(f"  {name:<30} r={r:.8f}  max_diff={max_diff:.6f}")

Testing candidate formulas against log_Hub:
  log10(CAGE_TPM)                r=0.39675022  max_diff=3.604810
  log10(CAGE_TPM + 1)            r=0.37758997  max_diff=2.910550
  log10(Cytosol_TPM)             r=0.78161218  max_diff=2.142530
  log10(Cytosol_TPM + 1)         r=0.80351772  max_diff=1.937019
  log10(PB_TPM)                  r=0.97890078  max_diff=2.162851
  log10(PB_TPM + 1)              r=0.99851651  max_diff=0.271438
  log10(PB_avg_CPM)              r=0.95913427  max_diff=2.135341
  log10(PB_avg_CPM + 1)          r=0.97508085  max_diff=0.954316
  log10(Cyt_avg_CPM)             r=0.82044588  max_diff=2.388389
  log10(Cyt_avg_CPM + 1)         r=0.85258394  max_diff=1.399086

Testing candidate formulas against log_FANTOM:
  log10(CAGE_TPM)                r=0.97982099  max_diff=1.027117
  log10(CAGE_TPM + 1)            r=1.00000000  max_diff=0.000000
  log10(Cytosol_TPM)             r=0.68337282  max_diff=3.530897
  log10(Cytosol_TPM + 1)         r=0.69452769  max_diff=3.53101

In [6]:
# Based on investigation above, compute the best-matching formulas
# If no formula matches, leave as NaN and document

# Placeholder — update after running the investigation cells above
# These may need to be: log10(PB_avg_CPM + 1) and log10(CAGE_TPM + 1), or similar
merged["log_Hub"] = np.nan  # TODO: set correct formula after investigation
merged["log_FANTOM"] = np.nan  # TODO: set correct formula after investigation
merged["delta_log10_Hub_minus_FANTOM"] = np.nan  # = log_Hub - log_FANTOM

print("NOTE: log_Hub/log_FANTOM formulas TBD after investigation.")
print("These columns are NOT used in the scatter plots (which use Cap_index).")

NOTE: log_Hub/log_FANTOM formulas TBD after investigation.
These columns are NOT used in the scatter plots (which use Cap_index).


## 4. Validation: Cap Index

In [7]:
# Compare our Cap_index against Jason's reference
val = merged[["ensembl_id", "Cap_index"]].merge(
    ref[["ensembl_id", "Cap_index"]],
    on="ensembl_id",
    how="inner",
    suffixes=("_ours", "_ref")
)

print(f"Genes compared: {len(val):,}")

mask = np.isfinite(val["Cap_index_ours"]) & np.isfinite(val["Cap_index_ref"])
r, _ = stats.pearsonr(val.loc[mask, "Cap_index_ours"], val.loc[mask, "Cap_index_ref"])
max_diff = (val.loc[mask, "Cap_index_ours"] - val.loc[mask, "Cap_index_ref"]).abs().max()
close = np.allclose(
    val.loc[mask, "Cap_index_ours"],
    val.loc[mask, "Cap_index_ref"],
    rtol=1e-5, atol=1e-8
)

print(f"\nCap_index comparison:")
print(f"  Pearson r: {r:.10f}")
print(f"  Max diff: {max_diff:.8f}")
print(f"  np.allclose(rtol=1e-5): {'PASS' if close else 'FAIL'}")

# Spot check GAPDH
gapdh = val[val["ensembl_id"].isin(ref[ref["Associated Gene Name"] == "GAPDH"]["ensembl_id"])]
if len(gapdh) > 0:
    row = gapdh.iloc[0]
    print(f"\n  GAPDH check: ours={row['Cap_index_ours']:.6f}, ref={row['Cap_index_ref']:.6f}")

Genes compared: 12,290

Cap_index comparison:
  Pearson r: 0.7224276787
  Max diff: 1555.95883795
  np.allclose(rtol=1e-5): FAIL

  GAPDH check: ours=1.658660, ref=1.629561


## 5. Save Output

In [8]:
output_path = PROCESSED_DIR / "07_hub_cage_with_cap_index.csv"
merged.to_csv(output_path, index=False)
print(f"Saved {len(merged):,} genes with Cap Index to {output_path}")

Saved 12,811 genes with Cap Index to ../data/processed/07_hub_cage_with_cap_index.csv
